In [4]:
import pandas as pd

# read annotation
gtf_path  = "/cluster-home/ziyanzha/annotation/gencode.v19.annotation.gtf.gz"
out_dir   = "/dbgap/martinzh_ukb16549/ziyanzha/Annotation"
tss_flank = 1000                                   # bp on each side of every TSS
chroms    = [f"chr{c}" for c in range(1, 23)]     

In [15]:
cols = ["chrom", "source", "feature", "start", "end", "score", "strand", "frame", "attr"]
gtf = pd.read_csv(gtf_path, sep="\t", comment="#", header=None, names=cols)

print(gtf.shape)
print(gtf["feature"].value_counts())
gtf.head(3)

(2619444, 9)
feature
exon              1196293
CDS                723784
UTR                284573
transcript         196520
start_codon         84144
stop_codon          76196
gene                57820
Selenocysteine        114
Name: count, dtype: int64


,chrom,source,feature,start,end,score,strand,frame,attr
0,chr1,HAVANA,gene,11869,14412,.,+,.,"gene_id ""ENSG00000223972.4""; transcript_id ""EN..."
1,chr1,HAVANA,transcript,11869,14409,.,+,.,"gene_id ""ENSG00000223972.4""; transcript_id ""EN..."
2,chr1,HAVANA,exon,11869,12227,.,+,.,"gene_id ""ENSG00000223972.4""; transcript_id ""EN..."


In [16]:
### only keep the exon and transcript
gtf = gtf[gtf["chrom"].isin(chroms) & gtf["feature"].isin(["exon", "transcript"])]

### extract the gene information
for key in ["gene_id", "gene_name", "gene_type", "transcript_id"]:
    gtf[key] = gtf["attr"].str.extract(rf'{key} "([^"]+)"', expand=False)
gtf = gtf.drop(columns=["source", "score", "frame", "attr"])

In [22]:
## get the exon region
exon = gtf.loc[gtf["feature"] == "exon",
               ["chrom", "start", "end", "gene_id", "gene_name", "gene_type"]]
## get TSS + flank region
tx = gtf[gtf["feature"] == "transcript"]
tss = tx["start"].where(tx["strand"] == "+", tx["end"])
prom = tx[["chrom", "gene_id", "gene_name", "gene_type"]].copy()
prom["start"] = (tss - tss_flank).clip(lower=1)
prom["end"]   = tss + tss_flank
prom = prom[exon.columns]                     

In [32]:
#combine two table
combine = pd.concat([exon, prom], ignore_index=True)
combine = combine.sort_values(["gene_id", "start", "end"]).reset_index(drop=True)

print("The number is correct?")
print(len(combine) == len(prom) + len(exon))

The number is correct?
True


In [37]:
#remove the duplicate SNP with one gene
run_end = combine.groupby("gene_id")["end"].cummax()
new_block = (combine["gene_id"] != combine["gene_id"].shift()) | \
            (combine["start"] > run_end.shift())
combine["block"] = new_block.cumsum()
regions = (combine.groupby("block")
                  .agg(chrom=("chrom", "first"), start=("start", "min"), end=("end", "max"),
                       gene_id=("gene_id", "first"), gene_name=("gene_name", "first"),
                       gene_type=("gene_type", "first"))
                  .reset_index(drop=True))

In [39]:
### store the data
regions["chrom"] = regions["chrom"].str.replace("chr", "", regex=False)
regions["chrom_num"] = regions["chrom"].astype(int)
regions = (regions.sort_values(["chrom_num", "start", "gene_id"])
                  .drop(columns="chrom_num")
                  .reset_index(drop=True))

out_path = f"{out_dir}/gene_regions_exon_tss1kb_gencode_v19.tsv"
regions.to_csv(out_path, sep="\t", index=False)
print("written:", out_path)

written: /dbgap/martinzh_ukb16549/ziyanzha/Annotation/gene_regions_exon_tss1kb_gencode_v19.tsv


,chrom,start,end,gene_id,gene_name,gene_type
0,1,10869,13052,ENSG00000223972.4,DDX11L1,pseudogene
1,1,13221,14412,ENSG00000223972.4,DDX11L1,pseudogene
2,1,14363,14829,ENSG00000227232.4,WASH7P,pseudogene
3,1,14970,15038,ENSG00000227232.4,WASH7P,pseudogene
4,1,15796,15947,ENSG00000227232.4,WASH7P,pseudogene


In [43]:
## simply check how many bp takes in genome
genome_bp = 2881033286          # GRCh37 autosomes (chr1-22), bp
total_bp = (regions["end"] - regions["start"] + 1).sum()
print(f"{total_bp/1e6:.1f} Mb, {total_bp/genome_bp:.2%} of autosomes")

276.1 Mb, 9.58% of autosomes
